# Silver Cleaning (B): istinad və xarici data

**Məsuliyyətli:** Aslan

| Bronze | Silver |
|---|---|
| `bronze.olist_products_dataset` | `silver.products` |
| `bronze.product_category_name_translation` | `silver.category_translation` |
| `bronze.olist_sellers_dataset` | `silver.sellers` |
| `bronze.olist_geolocation_dataset` | `silver.geolocation` |
| `bronze.olist_order_reviews_dataset` | `silver.reviews` |
| `bronze.api_exchange_rates` | `silver.exchange_rates` |
| `bronze.api_weather` | `silver.weather` |





> `silver.reviews` üçün `silver.orders` (A tərəfindən) hazır olmalıdır.

In [23]:
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# notebook-un olduğu qovluqdan bir yuxarı qovluq (db.py orada)
sys.path.append(str(Path.cwd().parent))

from db import get_engine

engine=get_engine()

In [24]:
"""SELECT sorğusu: nəticəni DataFrame kimi qaytarır."""
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

"""CREATE/DROP/INSERT: icra edir və commit edir."""
def run(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

### Silver. Products table

In [25]:
# Checking bronze products data before cleaning    

q("""
SELECT COUNT(*) AS setir,                                   -- ümumi sətir sayı
       COUNT(DISTINCT product_id) AS unikal_id,             -- unikal məhsul sayı (dublikat varmı?)
       SUM(CASE WHEN product_category_name = '' THEN 1 ELSE 0 END) AS bos_kateqoriya,  -- boş kateqoriyalar
       SUM(CASE WHEN product_weight_g = '' THEN 1 ELSE 0 END) AS bos_ceki              -- boş çəkilər
FROM bronze.olist_products_dataset
""")

,setir,unikal_id,bos_kateqoriya,bos_ceki
0,32951,32951,610,2


In [26]:
# Checking bronze products data types

q("""
    SELECT COLUMN_NAME,DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA='bronze' AND TABLE_NAME='olist_products_dataset'
""")

,COLUMN_NAME,DATA_TYPE
0,product_id,varchar
1,product_category_name,varchar
2,product_name_lenght,varchar
3,product_description_lenght,varchar
4,product_photos_qty,varchar
5,product_weight_g,varchar
6,product_length_cm,varchar
7,product_height_cm,varchar
8,product_width_cm,varchar


In [27]:
# We are deleting old table if available, to avoid errors when creating new table
run("DROP TABLE IF EXISTS silver.products")

# We are changing data types

run("""
-- 1-ci mərhələ (c): təmizləmə və tip çevirmələri
WITH c AS (
    SELECT
        NULLIF(TRIM(product_id), '')                                    AS product_id,
        LOWER(NULLIF(TRIM(product_category_name), ''))                  AS category_name,
        -- bronze-da "lenght" yazılış səhvi var, silver-də "length" edirik
        TRY_CAST(NULLIF(TRIM(product_name_lenght), '') AS INT)          AS name_length,
        TRY_CAST(NULLIF(TRIM(product_description_lenght), '') AS INT)   AS description_length,
        TRY_CAST(NULLIF(TRIM(product_photos_qty), '') AS INT)           AS photos_qty,
        TRY_CAST(NULLIF(TRIM(product_weight_g), '') AS DECIMAL(10,2))   AS weight_g,
        TRY_CAST(NULLIF(TRIM(product_length_cm), '') AS DECIMAL(10,2))  AS length_cm,
        TRY_CAST(NULLIF(TRIM(product_height_cm), '') AS DECIMAL(10,2))  AS height_cm,
        TRY_CAST(NULLIF(TRIM(product_width_cm), '') AS DECIMAL(10,2))   AS width_cm
    FROM bronze.olist_products_dataset
),

-- 2-ci mərhələ (d): dublikatları nömrələyirik
d AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY (SELECT NULL)) AS rn
    FROM c
    WHERE product_id IS NOT NULL
)

SELECT product_id, category_name, name_length, description_length, photos_qty,
       weight_g, length_cm, height_cm, width_cm
INTO silver.products
FROM d
WHERE rn = 1
""")

In [28]:
# checking silver prodcuts table after cleaning
q("""
SELECT TOP 5* FROM silver.products
""")

,product_id,category_name,name_length,description_length,photos_qty,weight_g,length_cm,height_cm,width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,53,596,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,56,752,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,50,266,2,300.0,45.0,15.0,35.0
3,001795ec6f1b187d37335e1c4704762e,consoles_games,53,274,1,600.0,30.0,20.0,20.0
4,001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,42,253,1,6000.0,40.0,4.0,30.0


### Silver. Category Translation table

In [29]:
# Check bronze category translation data before cleaning
q("""
SELECT COUNT(*) AS setir,                                                        -- ümumi sətir sayı
       COUNT(DISTINCT product_category_name) AS unikal_kateqoriya,               -- unikal portuqal adları (dublikat varmı?)
       SUM(CASE WHEN product_category_name = '' THEN 1 ELSE 0 END)         AS bos_pt,  -- boş portuqal adı
       SUM(CASE WHEN product_category_name_english = '' THEN 1 ELSE 0 END) AS bos_en   -- boş ingilis adı
FROM bronze.product_category_name_translation
""")

,setir,unikal_kateqoriya,bos_pt,bos_en
0,71,71,0,0


In [30]:
# Checking data types

q("""
    SELECT COLUMN_NAME,DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA='bronze' AND TABLE_NAME='product_category_name_translation'
""")

,COLUMN_NAME,DATA_TYPE
0,product_category_name,varchar
1,product_category_name_english,varchar


In [31]:
# Delete the old table if it exists, to avoid errors when creating the new one
run("DROP TABLE IF EXISTS silver.category_translation")

run("""
SELECT DISTINCT                                                          -- DISTINCT removes exact duplicates
    LOWER(NULLIF(TRIM(product_category_name), ''))         AS category_name,          -- Portuguese name
    LOWER(NULLIF(TRIM(product_category_name_english), '')) AS category_name_english   -- English name
INTO silver.category_translation
FROM bronze.product_category_name_translation
WHERE NULLIF(TRIM(product_category_name), '') IS NOT NULL                -- skip rows with an empty category name
""")

In [32]:
# checking silver category_translation table after cleaning
q("""
SELECT TOP 5* FROM silver.category_translation
""")

,category_name,category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


### Silver. Sellers Table

In [33]:
# Check the data types in the sellers table
q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_NAME = 'olist_sellers_dataset'
ORDER BY ORDINAL_POSITION
""")

,COLUMN_NAME,DATA_TYPE
0,seller_id,varchar
1,seller_zip_code_prefix,varchar
2,seller_city,varchar
3,seller_state,varchar


In [34]:
# Total rows vs. unique seller_id
q("""
SELECT COUNT(*)                  AS total_rows,
       COUNT(DISTINCT seller_id) AS unique_sellers
FROM bronze.olist_sellers_dataset
""")

,total_rows,unique_sellers
0,3095,3095


In [35]:
# Change data types in the sellers table to match the expected types in silver.sellers

run("DROP TABLE IF EXISTS silver.sellers")

run("""
WITH c AS (
    SELECT
        NULLIF(TRIM(seller_id), '')                                           AS seller_id,
        TRY_CAST(NULLIF(TRIM(seller_zip_code_prefix), '') AS INT)             AS zip_code_prefix,
        LOWER(NULLIF(TRIM(seller_city), ''))                                  AS city,
        UPPER(NULLIF(TRIM(seller_state), ''))                                 AS state
    FROM bronze.olist_sellers_dataset
),
d AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY (SELECT NULL)) AS rn
    FROM c WHERE seller_id IS NOT NULL
)
SELECT seller_id, zip_code_prefix, city, state
INTO silver.sellers
FROM d WHERE rn = 1
""")


In [36]:
# Checking silver table after cleaning
q("""
SELECT TOP 5* FROM silver.sellers
""")

,seller_id,zip_code_prefix,city,state
0,0015a82c2db000af6aaaf3ae2ecb0532,9080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP
4,010da0602d7774602cd1b3f5fb7b709e,9710,sao bernardo do campo,SP


### Silver. Geolocation table

In [37]:
# Check duplicates in the geolocation table
q("""
SELECT COUNT(*) AS cemi,
       COUNT(DISTINCT CONCAT(geolocation_zip_code_prefix, '|', geolocation_lat, '|',
                             geolocation_lng, '|', geolocation_city, '|', geolocation_state)) AS unikal
FROM bronze.olist_geolocation_dataset
""")

,cemi,unikal
0,1000163,738332


In [38]:
# Check the data types in the geolocation table
q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_NAME = 'olist_geolocation_dataset'
ORDER BY ORDINAL_POSITION
""")

,COLUMN_NAME,DATA_TYPE
0,geolocation_zip_code_prefix,varchar
1,geolocation_lat,varchar
2,geolocation_lng,varchar
3,geolocation_city,varchar
4,geolocation_state,varchar


In [41]:
# Step 1: cast types and standardize text (no deduplication yet)

run("DROP TABLE IF EXISTS silver.geolocation")

run("""
SELECT
    TRY_CAST(NULLIF(TRIM(geolocation_zip_code_prefix), '') AS INT)  AS zip_code_prefix,
    TRY_CAST(NULLIF(TRIM(geolocation_lat), '') AS DECIMAL(18,15))   AS lat,
    TRY_CAST(NULLIF(TRIM(geolocation_lng), '') AS DECIMAL(18,15))   AS lng,
    -- lowercase and strip accents (são paulo -> sao paulo)
    TRANSLATE(LOWER(NULLIF(TRIM(geolocation_city), '')),
              N'áàâãäéèêëíìîïóòôõöúùûüçñ',
              N'aaaaaeeeeiiiiooooouuuucn')                           AS city,
    UPPER(NULLIF(TRIM(geolocation_state), ''))                       AS state
INTO silver.geolocation
FROM bronze.olist_geolocation_dataset
""")

In [42]:
# Check: how many rows were removed
q("""
SELECT (SELECT COUNT(*) FROM bronze.olist_geolocation_dataset) AS bronze_rows,
       (SELECT COUNT(*) FROM silver.geolocation)               AS silver_rows
""")

,bronze_rows,silver_rows
0,1000163,1000163


In [43]:
# Checking silver table after cleaning
q("""
SELECT
    COLUMN_NAME,
    DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver'
  AND TABLE_NAME = 'geolocation';
""")

,COLUMN_NAME,DATA_TYPE
0,zip_code_prefix,int
1,lat,decimal
2,lng,decimal
3,city,nvarchar
4,state,varchar


### Silver. Reviews

In [ ]:
# Check bronze: is review_id unique?
q("""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT review_id) AS unique_review_id,
       COUNT(DISTINCT CONCAT(review_id, '|', order_id)) AS unique_review_order
FROM bronze.olist_order_reviews_dataset
""")

,row_count,unique_review_id,unique_review_order
0,99224,98410,99224


In [ ]:
run("DROP TABLE IF EXISTS silver.reviews")

run("""
-- Step 1 (c): cleaning and type casting
WITH c AS (
    SELECT
        NULLIF(TRIM(review_id), '')                                       AS review_id,
        NULLIF(TRIM(order_id), '')                                        AS order_id,
        TRY_CAST(NULLIF(TRIM(review_score), '') AS INT)                   AS review_score,     -- 1-5 rating
        NULLIF(TRIM(review_comment_title), '')                            AS comment_title,    -- empty -> NULL
        NULLIF(TRIM(review_comment_message), '')                          AS comment_message,  -- empty -> NULL
        TRY_CAST(NULLIF(TRIM(review_creation_date), '') AS DATETIME2)     AS creation_date,    -- text -> datetime
        TRY_CAST(NULLIF(TRIM(review_answer_timestamp), '') AS DATETIME2)  AS answer_timestamp
    FROM bronze.olist_order_reviews_dataset
),
-- Step 2 (d): duplicate key is review_id + order_id
d AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY review_id, order_id ORDER BY (SELECT NULL)) AS rn
    FROM c
    WHERE review_id IS NOT NULL
      AND order_id IS NOT NULL
      AND review_score BETWEEN 1 AND 5      -- keep valid scores only
)
SELECT review_id, order_id, review_score, comment_title, comment_message,
       creation_date, answer_timestamp
INTO silver.reviews
FROM d
WHERE rn = 1
""")

In [ ]:
# Check 1: row counts
q("""
SELECT (SELECT COUNT(*) FROM bronze.olist_order_reviews_dataset) AS bronze_rows,
       (SELECT COUNT(*) FROM silver.reviews)                     AS silver_rows
""")

,bronze_rows,silver_rows
0,99224,99224


In [ ]:
# Check 2: orphan reviews (needs silver.orders from person A; the join is for validation only)
q("""
SELECT COUNT(*) AS orphan_reviews
FROM silver.reviews r
LEFT JOIN silver.orders o ON o.order_id = r.order_id
WHERE o.order_id IS NULL
""")

,orphan_reviews
0,0


In [ ]:
# Check 2: orphan reviews (needs silver.orders from person A; the join is for validation only)
q("""
SELECT COUNT(*) AS orphan_reviews
FROM silver.reviews r
LEFT JOIN silver.orders o ON o.order_id = r.order_id
WHERE o.order_id IS NULL
""")

,orphan_reviews
0,0


### Silver. Exchange rates

In [ ]:
# 1. Check bronze: column types and first 5 rows
q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze' AND TABLE_NAME = 'api_exchange_rates'
ORDER BY ORDINAL_POSITION
""")

,COLUMN_NAME,DATA_TYPE
0,date,varchar
1,rate_usd,float


In [ ]:
run("DROP TABLE IF EXISTS silver.exchange_rates")

run("""
-- Cast date and rate; rate_usd is already float in bronze, so no TRIM is needed
WITH c AS (
    SELECT
        TRY_CAST(NULLIF(TRIM(CAST([date] AS VARCHAR(50))), '') AS DATE) AS rate_date,
        TRY_CAST(rate_usd AS DECIMAL(18,6))                            AS rate_usd   -- USD per 1 BRL
    FROM bronze.api_exchange_rates
),
d AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY rate_date ORDER BY (SELECT NULL)) AS rn
    FROM c
    WHERE rate_date IS NOT NULL AND rate_usd > 0     -- rate must be positive
)
SELECT rate_date, rate_usd
INTO silver.exchange_rates
FROM d
WHERE rn = 1
""")

In [ ]:
# 3. Check silver: first 5 rows
q("SELECT TOP 5 * FROM silver.exchange_rates ORDER BY rate_date")

,rate_date,rate_usd
0,2015-12-31,0.25250
1,2016-01-04,0.24755
2,2016-01-05,0.24977
3,2016-01-06,0.24791
4,2016-01-07,0.24699


### Silvers. Weather

In [ ]:
# 1. Check bronze: column types and first 5 rows
q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze' AND TABLE_NAME = 'api_weather'
ORDER BY ORDINAL_POSITION
""")

,COLUMN_NAME,DATA_TYPE
0,date,varchar
1,temp_max,float
2,temp_min,float
3,precipitation,float


In [ ]:
q("SELECT TOP 5 * FROM bronze.api_weather")

,date,temp_max,temp_min,precipitation
0,2016-01-01,27.7,21.0,9.1
1,2016-01-02,24.5,19.8,25.7
2,2016-01-03,23.2,18.9,4.4
3,2016-01-04,25.4,17.7,0.0
4,2016-01-05,26.4,16.8,0.0


In [ ]:
# 2. Clean and load into silver
run("DROP TABLE IF EXISTS silver.weather")

run("""
-- Step 1 (c): cast date and measurements (numeric columns are already float in bronze, so no TRIM)
WITH c AS (
    SELECT
        TRY_CAST(NULLIF(TRIM(CAST([date] AS VARCHAR(50))), '') AS DATE) AS weather_date,
        TRY_CAST(temp_max AS DECIMAL(6,2))                             AS temp_max,       -- max temperature (°C)
        TRY_CAST(temp_min AS DECIMAL(6,2))                             AS temp_min,       -- min temperature (°C)
        TRY_CAST(precipitation AS DECIMAL(8,2))                        AS precipitation   -- precipitation (mm)
    FROM bronze.api_weather
),
-- Step 2 (d): number duplicates by date
d AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY weather_date ORDER BY (SELECT NULL)) AS rn
    FROM c
    WHERE weather_date IS NOT NULL      -- drop rows without a date
)
SELECT weather_date, temp_max, temp_min, precipitation
INTO silver.weather
FROM d
WHERE rn = 1                            -- keep one row per date
""")

In [ ]:
# 3. Check silver: first 5 rows
q("SELECT TOP 5 * FROM silver.weather ORDER BY weather_date")

,weather_date,temp_max,temp_min,precipitation
0,2016-01-01,27.7,21.0,9.1
1,2016-01-02,24.5,19.8,25.7
2,2016-01-03,23.2,18.9,4.4
3,2016-01-04,25.4,17.7,0.0
4,2016-01-05,26.4,16.8,0.0
